# ChestXRayMaxViT-v2 Training (Colab)

Runs the enhanced NIH ChestX-ray14 classifier from `docs/enhanced_architecture_spec.md` / `docs/training_strategy.md`.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine, A100/L4 is faster).

**One-time setup you need outside this notebook:**
1. A free Kaggle account + API token: kaggle.com -> Settings -> Create New Token (downloads `kaggle.json`).
2. In this Colab notebook, click the key icon (Secrets) in the left sidebar and add two secrets: `KAGGLE_USERNAME` and `KAGGLE_KEY` (from `kaggle.json`). This avoids re-uploading credentials every session.
3. Replace `REPO_URL` in the cell below with your GitHub repo URL.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('torch', torch.__version__, '| torchvision', end=' ')
import torchvision; print(torchvision.__version__)

## 1. Get the code

In [ ]:
REPO_URL = "REPLACE_WITH_YOUR_GITHUB_REPO_URL"  # e.g. https://github.com/<you>/ChestXRayMaxVit.git

!git clone $REPO_URL /content/ChestXRayMaxVit
%cd /content/ChestXRayMaxVit
!pip install -q -r requirements-colab.txt

## 2. Mount Google Drive (for checkpoints that survive a disconnected session)

Checkpoints go to Drive, not the Colab VM's local disk -- the VM (and everything on it) is wiped when the session ends, but the dataset is re-downloaded fresh each session anyway (see step 3), so only checkpoints/logs need to persist.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/ChestXRayMaxViTv2/checkpoints'
LOG_DIR = '/content/drive/MyDrive/ChestXRayMaxViTv2/logs'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

## 3. Download the dataset from Kaggle

Downloads straight onto the Colab VM's fast local disk (not Drive -- Drive is much slower for the random-access image reads a DataLoader does). This is the same `nih-chest-xrays/data` dataset as the local `archive/` folder (same file names: `Data_Entry_2017.csv`, `images_001/images/` ... `images_012/images/`), so no code changes are needed, only the `--archive-dir` path.

In [ ]:
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!mkdir -p /content/archive
!kaggle datasets download -d nih-chest-xrays/data -p /content/archive --unzip

In [ ]:
# Sanity-check the download landed in the expected layout.
!ls /content/archive | head -20
!test -f /content/archive/Data_Entry_2017.csv && echo 'Data_Entry_2017.csv found'
!ls /content/archive/images_001/images | head -3

## 4. Pipeline sanity check (recommended before the full run)

A few epochs on a small subset -- confirms the full pipeline (data loading, model, loss, GPU training loop) works on this environment before committing GPU-hours to the full 100-epoch schedule.

In [ ]:
!python sanity_check.py --archive-dir /content/archive --num-workers 2

## 5. Full training run

Per `docs/training_strategy.md`: AdamW + cosine schedule with warmup, focal loss, early stopping (patience 10) on validation mean per-class F1, up to 100 epochs. Checkpoints (`last_checkpoint.pt` every epoch, `best_model.pt` on improvement) go to Drive.

In [ ]:
!python train.py \
  --archive-dir /content/archive \
  --checkpoint-dir "$CHECKPOINT_DIR" \
  --log-dir "$LOG_DIR" \
  --num-workers 2 \
  --device cuda

## 6. Resuming after a disconnected session

Colab free-tier sessions disconnect after ~12h (or sooner if idle). Re-run steps 1-3 (code + dataset are gone when the VM is recycled -- Drive is not), then resume from the last checkpoint instead of restarting from epoch 0:

In [ ]:
!python train.py \
  --archive-dir /content/archive \
  --checkpoint-dir "$CHECKPOINT_DIR" \
  --log-dir "$LOG_DIR" \
  --num-workers 2 \
  --device cuda \
  --resume "$CHECKPOINT_DIR/last_checkpoint.pt"